# 5.7 — Train MuscleMimic from scratch with FastTD3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/tutorial/fasttd3-directional-scratch/tutorials/5.7_FastTD3.ipynb)

Tutorial 5.3 trains a MuscleMimic walker with **on-policy PPO**. This notebook trains the same full-body model **from scratch** with **off-policy FastTD3** (Seo et al., 2025): no motion clip, no Hugging Face token, no behaviour cloning.

The task is `myoMimicFullbody-v0` (354 muscles, random site targets). FastTD3's actor emits `tanh` actions in `[-1, 1]`; the env wrapper maps them to muscle activations in `[0, 1]`.

## Flow

1. Open the env and inspect observation / action sizes.
2. Smoke-train on CPU (a few steps, tiny nets).
3. Train on CPU until the 6-muscle elbow pose is solved — proof FastTD3 learns without a GPU.
4. Optionally train the 354-muscle full-body env on GPU.
5. Render a rollout of the full-body checkpoint.

The CPU smoke is plumbing. The elbow cell is the learning check. A 50k-step T4 full-body run is a demo, not a finished walker.

Local install, from a MyoSuite clone:

```bash
pip install -e ".[musclemimic]"
pip install -e training/fasttd3
```

Colab: run **5.7.0** once, then continue.

> **Citation:** FastTD3 — [younggyoseo/FastTD3](https://github.com/younggyoseo/FastTD3) (Seo et al., 2025, MIT). MuscleMimic — [Li et al., 2026](https://arxiv.org/abs/2603.25544). This notebook does not download AMASS/KIT clips. See the MyoSuite *References* page.

## 5.7.0 — Colab install

Skip this cell if you already installed MyoSuite locally. It only clones the repo and installs the MuscleMimic extra plus the FastTD3 trainer.

In [ ]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import os
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYOPENGL_PLATFORM"] = "egl"
    !apt-get -qq install -y xvfb libegl1 libgles2 libosmesa6 libgl1
    %cd /content
    if not Path("/content/myosuite/myosuite").exists():
        !git clone --depth 1 -b tutorial/fasttd3-directional-scratch https://github.com/MyoHub/myosuite.git
    else:
        !git -C /content/myosuite fetch --depth 1 origin tutorial/fasttd3-directional-scratch
        !git -C /content/myosuite checkout -f FETCH_HEAD
    %cd /content/myosuite
    import torch
    if torch.cuda.is_available():
        # mjlab 1.6 == mujoco 3.11 + mujoco-warp 3.11 (3.17 conflicts with mjlab).
        %pip install -q "mujoco>=3.11,<3.12" "mujoco-warp>=3.11,<3.12" "warp-lang>=1.14" "mjlab>=1.6,<1.7"
        %pip install -q --upgrade-strategy only-if-needed -e ".[musclemimic,mjlab]"
    else:
        %pip install -q --upgrade-strategy only-if-needed -e ".[musclemimic]"
    %pip install -q --upgrade-strategy only-if-needed -e "./training/fasttd3"
else:
    print("Local runtime — skip Colab install.")

## 5.7.1 — Config

`GPU_TRAIN` stays off until the runtime has a GPU (Colab: Runtime → Change runtime type → T4).

In [ ]:
from pathlib import Path
import os
import sys

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ.setdefault("JAX_PLATFORMS", "cpu")

ENV_ID = "myoMimicFullbody-v0"
GPU_TRAIN = False
TRAIN_STEPS = 50_000
N_ENVS = 64
OUT = Path("runs/fasttd3")

for d in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (d / "myosuite" / "__init__.py").is_file():
        sys.path[:0] = [str(d), str(d / "training" / "fasttd3")]
        break

print(f"{ENV_ID}  GPU_TRAIN={GPU_TRAIN}  steps={TRAIN_STEPS}  envs={N_ENVS}")

## 5.7.2 — The environment

`myoMimicFullbody-v0` is the MuscleMimic full-body model (`musclemimic_models` 1.0.6) with random site targets — the same skeleton as ChaseTag and tutorial 5.4, without gated KIT clips.

Observation is about 684 values (qpos, qvel, activations, sites, target). Action is 354 muscle excitations in `[0, 1]`.

In [ ]:
from myosuite.core.registry import make_env

env = make_env(ENV_ID)
obs, _ = env.reset(seed=0)
print("observation", obs.shape)
print("action     ", env.action_space.shape, env.action_space.low.min(), "…", env.action_space.high.max())
env.close()

## 5.7.3 — FastTD3

The trainer lives under `training/fasttd3/` so method-specific dependencies stay out of the installable `myosuite` package. It uses the published FastTD3 actor, distributional critic, and replay buffer.

```python
from myosuite_fasttd3 import TrainArgs, train
```

AMP stays off on T4 (no bf16; fp16 overflows on this observation). Critic support is `[-250, 250]`.

## 5.7.4 — CPU smoke

Two environments, eight steps, small networks. This checks the loop; it will not learn to walk. You should see `backend=cpu`, `amp=False`, and a saved `smoke.pt`.

In [ ]:
from myosuite_fasttd3 import TrainArgs, train

ckpt = train(TrainArgs(env_id=ENV_ID, smoke=True, backend="cpu", device="cpu", out=OUT))
print("checkpoint:", ckpt)

## 5.7.5 — CPU train (learns)

Same FastTD3 loop on `myoElbowPose1D6MRandom-v0` (6 muscles). About a minute on a laptop CPU. `ema_r` should climb from around 0–2 toward **7**, and `qloss` should fall. That is the check that the update learns — the 354-muscle full-body env needs the GPU cell.

Full-body from scratch on CPU does run, but a short laptop run does not beat random site tracking.

In [ ]:
from myosuite_fasttd3 import TrainArgs, train

cpu_ckpt = train(
    TrainArgs(
        env_id="myoElbowPose1D6MRandom-v0",
        backend="cpu",
        device="cpu",
        num_envs=8,
        total_timesteps=4_000,
        batch_size=256,
        buffer_size=2048,
        learning_starts=32,
        actor_hidden=128,
        critic_hidden=128,
        num_atoms=21,
        out=OUT / "cpu_elbow",
    )
)
print("checkpoint:", cpu_ckpt)

## 5.7.6 — GPU training (optional)

Same env id, mjlab Warp backend. On Colab, switch to a **T4** runtime, rerun from 5.7.0, set `GPU_TRAIN = True`, then run this cell.

The Warp stack is **mjlab 1.6 + MuJoCo 3.11 + mujoco-warp 3.11**. Do not install MuJoCo 3.13+; mjlab cannot resolve that. If this session already imported another `mujoco`, **Restart session** after 5.7.0, then continue.

Log fields: `qloss`, `q` (should stay in roughly `[-250, 250]`), `pi`, `r` (step reward, about 0–1 here).

In [ ]:
import importlib.util
import torch
from myosuite_fasttd3 import TrainArgs, train

def _mujoco_ready_for_warp():
    import mujoco
    model = mujoco.MjModel.from_xml_string("<mujoco><worldbody/></mujoco>")
    return hasattr(model, "nflexevpair"), mujoco.__version__

if not GPU_TRAIN:
    print("GPU_TRAIN is False — skip. Enable it on a T4 runtime after the CPU cells.")
elif not torch.cuda.is_available():
    print("No GPU in this session. Colab: Runtime → Change runtime type → T4, then rerun.")
else:
    ready, mj_ver = _mujoco_ready_for_warp()
    if importlib.util.find_spec("mjlab") is None or not ready:
        %pip install -q "mujoco>=3.11,<3.12" "mujoco-warp>=3.11,<3.12" "warp-lang>=1.14" "mjlab>=1.6,<1.7"
        ready, mj_ver = _mujoco_ready_for_warp()
        if not ready:
            raise SystemExit(
                f"mujoco {mj_ver} is still the previously loaded binary. "
                "Runtime → Restart session, then rerun from 5.7.0 with GPU_TRAIN = True."
            )
    ckpt = train(
        TrainArgs(
            env_id=ENV_ID,
            backend="mjlab",
            num_envs=N_ENVS,
            total_timesteps=TRAIN_STEPS,
            out=OUT,
        )
    )
    print("checkpoint:", ckpt)

## 5.7.7 — Render

Play the latest **full-body** checkpoint on the CPU env and embed an MP4. After the smoke cell this is a short untrained clip; after GPU training it is the policy you just trained.

Colab has no display, and a failed GLFW init in this kernel cannot be recovered. The clip is recorded in a **fresh process** under Xvfb.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from IPython.display import Video, display

final = OUT / f"{ENV_ID}_final.pt"
smoke = OUT / f"{ENV_ID}_smoke.pt"
ckpt_path = final if final.is_file() else smoke
if not ckpt_path.is_file():
    raise FileNotFoundError(f"No checkpoint in {OUT} — run 5.7.4 or 5.7.6 first.")

OUT.mkdir(parents=True, exist_ok=True)
video_path = OUT / "rollout.mp4"
script = Path("/tmp/fasttd3_render_rollout.py")
script.write_text('"""Render a FastTD3 checkpoint to MP4. Safe under xvfb-run (Colab has no DISPLAY)."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport sys\nfrom pathlib import Path\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--ckpt", required=True)\n    parser.add_argument("--out", required=True)\n    parser.add_argument("--env", default="myoMimicFullbody-v0")\n    parser.add_argument("--steps", type=int, default=200)\n    args = parser.parse_args()\n\n    if not os.environ.get("MUJOCO_GL"):\n        os.environ["MUJOCO_GL"] = (\n            "glfw" if sys.platform == "darwin" or os.environ.get("DISPLAY") else "egl"\n        )\n\n    import imageio\n    import mujoco\n    import numpy as np\n    import torch\n    from fast_td3.fast_td3 import Actor\n    from fast_td3.fast_td3_utils import EmpiricalNormalization\n    from myosuite.core.registry import make_env\n\n    ckpt_path = Path(args.ckpt)\n    blob = torch.load(ckpt_path, map_location="cpu", weights_only=True)\n    n_obs, n_act = int(blob["n_obs"]), int(blob["n_act"])\n    hidden = int(blob["actor"]["net.0.weight"].shape[0])\n    n_train_envs = int(blob["actor"]["noise_scales"].shape[0])\n    actor = Actor(\n        n_obs=n_obs,\n        n_act=n_act,\n        num_envs=n_train_envs,\n        init_scale=0.01,\n        hidden_dim=hidden,\n        device=torch.device("cpu"),\n    )\n    actor.load_state_dict(blob["actor"])\n    actor.eval()\n    obs_norm = EmpiricalNormalization(shape=n_obs, device=torch.device("cpu"))\n    norm_state = blob.get("obs_norm")\n    if isinstance(norm_state, dict) and norm_state:\n        obs_norm.load_state_dict(norm_state)\n    obs_norm.eval()\n\n    env = make_env(args.env)\n    obs, _ = env.reset(seed=0)\n    raw = env.unwrapped\n    fps = round(1.0 / float(getattr(raw, "_ctrl_dt", 0.01)))\n    renderer = mujoco.Renderer(raw.model, height=320, width=420)\n    cam = mujoco.MjvCamera()\n    cam.type = mujoco.mjtCamera.mjCAMERA_FREE\n    cam.lookat[:] = [0.0, 0.0, 1.0]\n    cam.distance, cam.azimuth, cam.elevation = 3.5, 135.0, -20.0\n\n    frames = []\n    for _ in range(args.steps):\n        x = torch.as_tensor(np.asarray(obs, np.float32)).reshape(1, -1)\n        if int(x.shape[-1]) != n_obs:\n            raise ValueError(f"obs dim {x.shape[-1]} != checkpoint n_obs {n_obs}")\n        with torch.no_grad():\n            tanh_act = actor(obs_norm(x, update=False)).cpu().numpy()[0]\n        muscle = (np.clip(tanh_act, -1.0, 1.0) + 1.0) * 0.5\n        obs, _, terminated, truncated, _ = env.step(muscle)\n        if raw.data.qpos.size > 2:\n            cam.lookat[:2] = raw.data.qpos[:2]\n        renderer.update_scene(raw.data, camera=cam)\n        frames.append(np.asarray(renderer.render(), dtype=np.uint8).copy())\n        if terminated or truncated:\n            break\n    renderer.close()\n    env.close()\n\n    out = Path(args.out)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with imageio.get_writer(str(out), fps=fps, quality=8, macro_block_size=1) as writer:\n        for frame in frames:\n            writer.append_data(frame)\n    print(f"{ckpt_path.name}: {len(frames)} frames → {out}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')

cmd = [
    sys.executable,
    str(script),
    "--ckpt",
    str(ckpt_path.resolve()),
    "--env",
    ENV_ID,
    "--out",
    str(video_path.resolve()),
]
env = os.environ.copy()
env["MUJOCO_GL"] = "glfw"
env.pop("PYOPENGL_PLATFORM", None)
if "google.colab" in sys.modules:
    subprocess.run(["apt-get", "-qq", "install", "-y", "xvfb"], check=False)
    cmd = ["xvfb-run", "-a", "-s", "-screen 0 1024x768x24", *cmd]
subprocess.check_call(cmd, env=env)
print(f"{ckpt_path.name} → {video_path}")
display(Video(str(video_path), embed=True, html_attributes="controls loop"))


## 5.7.8 — Next

- Larger GPU run: `train(TrainArgs(env_id=ENV_ID, backend="mjlab", num_envs=1024, total_timesteps=150_000))`
- Heading-conditioned walking with a teacher: tutorial **5.4**
- ChaseTag submission: tutorial **5.6**
- Trainer notes: `training/fasttd3/README.md`